# Agentic RAG: 에이전트 기반 RAG - 실습 코드 1: Agentic RAG 구현 (간략화)

- Tutorial ID: `expand-agentic-rag`
- Tutorial: Agentic RAG: 에이전트 기반 RAG
- Section ID: `expand-agentic-rag-code-1`
- Section: 실습 코드 1: Agentic RAG 구현 (간략화)


In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 1: Agentic RAG 구현 (간략화)
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#   2) 질문 → 검색 → 근거 선택 → 답변/검증 파이프라인을 단계별로 확인
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
from enum import Enum
from typing import List, Dict
import json

class SearchStrategy(Enum):
    SEMANTIC = "semantic"
        MULTI_QUERY = "multi_query"
    HYBRID = "hybrid"

class AgenticRAG:
        def __init__(self, llm, vectorstore):
        self.llm = llm
        self.vectorstore = vectorstore
        self.max_iterations = 3
    
        def should_retrieve(self, query: str) -> bool:
        """검색 필요성 판단"""
        prompt = f"Does this question require external knowledge? '{query}' Answer yes/no."
        response = self.llm(prompt)
        return "yes" in response.lower()
    
        def select_strategy(self, query: str) -> SearchStrategy:
        """검색 전략 선택"""
        prompt = f"""Select search strategy for: '{query}'
        Options: semantic, multi_query, hybrid
        Consider: factual=semantic, complex=multi_query, mixed=hybrid"""
        response = self.llm(prompt)
        return SearchStrategy(response.strip().lower())
    
        def retrieve(self, query: str, strategy: SearchStrategy) -> List[Dict]:
        """전략에 따른 검색"""
        if strategy == SearchStrategy.SEMANTIC:
            return self.vectorstore.similarity_search(query, k=3)
        elif strategy == SearchStrategy.MULTI_QUERY:
            queries = self._generate_multi_queries(query)
            results = []
                        for q in queries:
                results.extend(self.vectorstore.similarity_search(q, k=2))
            return results
        elif strategy == SearchStrategy.HYBRID:
            semantic = self.vectorstore.similarity_search(query, k=2)
            keyword = self.vectorstore.keyword_search(query, k=2)
            return semantic + keyword
    
        def query(self, question: str) -> str:
        """전체 Agentic RAG 파이프라인"""
        if not self.should_retrieve(question):
            return self.llm(question)
        
        strategy = self.select_strategy(question)
        context = self.retrieve(question, strategy)
        
        prompt = f"""Based on context, answer the question.
        Context: {context}
        Question: {question}
        Answer:"""
        return self.llm(prompt)